In [ ]:
%matplotlib inline


# Imputation

Not every sample is assayed for every variable. In the stacked sulphide lenses, density was measured on about half
of the diamond core from the sulphide units and never on RC chips, and the RC holes were not assayed for gold. The
grades Zn, Pb, Cu and Ag are complete. `GaussianImputer` fills the gaps: it normal-scores each variable on its own
values, fits the correlation of the scores to all rows, and draws each missing score given the scores present in
its row.


In [ ]:
import sys
from pathlib import Path

HERE = Path(__file__).parent if "__file__" in globals() else Path.cwd()
sys.path.insert(0, str(HERE.parents[1]))

In [ ]:
import ceres as cs
import matplotlib.pyplot as plt
import numpy as np
from common import ACCENT, GRAY, HIGHLIGHT, LIGHT, save

lenses = cs.datasets.stacked_sulphide_lenses()
samples = cs.merge_intervals(lenses["assays"], lenses["lithology"])
sulphide = np.isin(np.asarray(samples["LITH"]), ["MS", "SMS", "STR"]) & ~np.isnan(samples["ZN_PCT"])
columns = ["ZN_PCT", "PB_PCT", "CU_PCT", "AG_GPT", "AU_GPT", "DENSITY"]
labels = ["Zn", "Pb", "Cu", "Ag", "Au", "density"]
data = np.column_stack([samples[c][sulphide] for c in columns])
missing = np.isnan(data)
print(
    f"{len(data)} sulphide samples, missing: "
    + ", ".join(f"{n} {m.sum()}" for n, m in zip(labels, missing.T))
)

The scores of density follow Zn, Pb and Ag closely; gold is weakly tied to the others, so its drawn values will
carry more of their own randomness.



In [ ]:
imputer = cs.GaussianImputer(seed=0).fit(data)
print("score correlation")
print(" " * 8 + "".join(f"{n:>8}" for n in labels))
for name, row in zip(labels, imputer.correlation_):
    print(f"{name:8}" + "".join(f"{r:8.2f}" for r in row))

To check the method, hide the measured density of every other hole, impute it, and compare with the truth: one draw,
and the mean of 50 draws with different seeds, close to the conditional mean.



In [ ]:
holes = np.asarray(samples["HOLE_ID"])[sulphide]
test = ~missing[:, 5] & np.isin(holes, np.unique(holes)[::2])
holed = data.copy()
holed[test, 5] = np.nan
draws = np.array([cs.GaussianImputer(seed=s).fit(holed).transform(holed)[test, 5] for s in range(50)])
truth = data[test, 5]
qs = [0.1, 0.5, 0.9]
print(f"{test.sum()} densities hidden")
for name, x in {"truth": truth, "one draw": draws[0], "mean of 50": draws.mean(axis=0)}.items():
    q = ", ".join(f"{v:.2f}" for v in np.quantile(x, qs))
    rmse = np.sqrt(np.mean((x - truth) ** 2))
    print(f"{name:11} q10, q50, q90: {q}   sd {x.std():.2f}" + (f"   rmse {rmse:.2f}" if rmse else ""))

fig, axes = plt.subplots(1, 2, figsize=(8, 3.6), layout="constrained", sharex=True, sharey=True)
for ax, (x, title) in zip(axes, [(draws[0], "One draw"), (draws.mean(axis=0), "Mean of 50 draws")]):
    ax.scatter(truth, x, s=4, color=ACCENT, alpha=0.4, linewidths=0)
    ax.axline((3, 3), slope=1, color=GRAY, lw=0.8)
    ax.set(xlabel="Measured density (t/m³)", title=title, aspect="equal")
axes[0].set_ylabel("Imputed density (t/m³)")
save(fig, "check")

One draw keeps the spread of the hidden densities (sd 0.37 against 0.36), with a root-mean-square error of
0.21 t/m³. The mean of 50 draws cuts the error to 0.13 but narrows the spread to 0.32, like any prediction. Draws
suit simulation and tonnage uncertainty; the mean suits a single best value per sample.

Filling every gap with one draw: imputed densities follow the trend of the measured ones against zinc, with the same
scatter.



In [ ]:
filled = imputer.transform(data)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6), layout="constrained")
for ax, j, title, label in ((axes[0], 5, "Density", "t/m³"), (axes[1], 4, "Gold, missing in RC", "Au (g/t)")):
    gap = missing[:, j]
    ax.scatter(data[~gap, 0], data[~gap, j], s=3, color=LIGHT, linewidths=0, label="measured")
    ax.scatter(filled[gap, 0], filled[gap, j], s=3, color=HIGHLIGHT, alpha=0.5, linewidths=0, label="imputed")
    ax.set(xlabel="Zn (%)", ylabel=label, xscale="log", title=title)
axes[1].set_yscale("log")
axes[0].legend(markerscale=3)
save(fig, "filled")
print(f"density mean {np.nanmean(data[:, 5]):.3f} measured, {filled[:, 5].mean():.3f} after imputation")

The imputer assumes the missing values behave like the measured ones with the same grades. That holds for density
here, measured on part of the same core; gold in RC chips borrows its relation to the base metals from the diamond
holes. `MultivariateSimulation.fit(..., impute=True)` redraws the gaps in every realization ([multivariate simulation](../../08-stochastic-simulation/06-multivariate-simulation/README.md)).

